# 01. Exploratory Data Analysis: PURE Dataset

This notebook analyzes the authentic PURE (Public REquirements) dataset (Ferrari et al., RE 2017).
Per CheckSense binding rules, all statistics and samples are drawn directly from `data/interim/pure_requirements.jsonl`.


In [1]:
import json
from pathlib import Path

import pandas as pd

# 1. Load interim requirements
jsonl_path = Path('../data/interim/pure_requirements.jsonl')
records = [json.loads(line) for line in jsonl_path.read_text(encoding='utf-8').splitlines() if line.strip()]
df = pd.DataFrame(records)
print(f'Total candidate requirements: {len(df):,}')
print(f'Unique documents: {df["doc_id"].nunique()}')
df[['req_id', 'doc_id', 'token_count', 'source_file']].head()


Total candidate requirements: 2,928
Unique documents: 18


,req_id,doc_id,token_count,source_file
0,0000 - cctns_sent_0000,0000 - cctns,22,0000 - cctns.xml
1,0000 - cctns_sent_0002,0000 - cctns,22,0000 - cctns.xml
2,0000 - cctns_sent_0003,0000 - cctns,25,0000 - cctns.xml
3,0000 - cctns_sent_0004,0000 - cctns,14,0000 - cctns.xml
4,0000 - cctns_sent_0005,0000 - cctns,23,0000 - cctns.xml


## 2. Requirements per Document Breakdown


In [2]:
doc_counts = df['doc_id'].value_counts().reset_index()
doc_counts.columns = ['Document ID', 'Requirement Count']
doc_counts


,Document ID,Requirement Count
0,2007-eirene_fun_7-2,508
1,2009 - peppol approved,470
2,1995 - gemini,366
3,2006 - eirene sys 15,361
4,2007-ertms,203
5,1999 - tcs,178
6,2005 - phin,139
7,0000 - cctns,128
8,1998 - themas,112
9,2005 - microcare,111


## 3. Token Length Distribution


In [3]:
stats = df['token_count'].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])
print('Token Count Summary Statistics:')
print(stats)



Token Count Summary Statistics:
count    2928.000000
mean       23.019809
std        10.564828
min         5.000000
25%        15.000000
50%        21.000000
75%        29.000000
90%        37.000000
95%        44.000000
max        60.000000
Name: token_count, dtype: float64


## 4. Modal Verb Proportions


In [4]:
import re

modals = {'shall': r'\bshall\b', 'must': r'\bmust\b', 'should': r'\bshould\b', 'will': r'\bwill\b'}
modal_counts = {}
for name, pat in modals.items():
    count = df['text'].str.contains(pat, flags=re.IGNORECASE).sum()
    modal_counts[name] = {'count': count, 'percentage': round(100 * count / len(df), 2)}
pd.DataFrame(modal_counts).T


,count,percentage
shall,1293.0,44.16
must,488.0,16.67
should,526.0,17.96
will,732.0,25.00


## 5. Filtering Audit Statistics


In [5]:
stats_path = Path('../data/interim/pure_filtering_stats.json')
filter_stats = json.loads(stats_path.read_text(encoding='utf-8'))
pd.DataFrame(list(filter_stats['summary'].items()), columns=['Metric', 'Value'])


,Metric,Value
0,total_documents_processed,18.0000
1,documents_dropped,0.0000
2,total_raw_sentences_extracted,7273.0000
3,removed_duplicates,408.0000
4,removed_headings,45.0000
5,removed_no_modal_verb,3749.0000
6,removed_too_short_lt5,0.0000
7,removed_too_long_gt60,143.0000
8,total_candidate_requirements_retained,2928.0000
9,modal_filter_retention_rate,0.4265


## 6. Twenty Random Requirement Metadata Samples (Seed 42)


In [6]:
sample_20 = df.sample(n=20, random_state=42)[['req_id', 'doc_id', 'token_count', 'source_file']].reset_index(drop=True)
sample_20


,req_id,doc_id,token_count,source_file
0,2009 - peppol approved_sent_0278,2009 - peppol approved,8,2009 - peppol approved.xml
1,0000 - gamma j_sent_0109,0000 - gamma j,15,0000 - gamma j.xml
2,2008 - peering_sent_0004,2008 - peering,12,2008 - peering.xml
3,2007-eirene_fun_7-2_sent_0252,2007-eirene_fun_7-2,28,2007-eirene_fun_7-2.xml
4,2005 - phin_sent_0086,2005 - phin,27,2005 - phin.xml
5,2006 - eirene sys 15_sent_0123,2006 - eirene sys 15,57,2006 - eirene sys 15.xml
6,2006 - eirene sys 15_sent_0650,2006 - eirene sys 15,15,2006 - eirene sys 15.xml
7,2005 - phin_sent_0164,2005 - phin,41,2005 - phin.xml
8,1995 - gemini_sent_0277,1995 - gemini,21,1995 - gemini.xml
9,2006 - eirene sys 15_sent_0368,2006 - eirene sys 15,14,2006 - eirene sys 15.xml
